<a href="https://colab.research.google.com/github/BakrAdli/My_AI_Journey/blob/main/smart_car_diagnostics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ========================================================
# GARAGE MANAGEMENT SYSTEM V3.1 (AI Memory & Resilience)
# ========================================================
import json
import os
import time

# --- 1. AI INITIALIZATION & MEMORY SETUP ---
LLM_READY = False
chat_session = None

try:
    import google.generativeai as genai
    from google.colab import userdata

    # Fetch API Key securely
    api_key = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=api_key)

    # Initialize model with System Prompt
    model_name = 'gemini-3-flash-preview'
    model = genai.GenerativeModel(
        model_name,
        system_instruction="You are an expert AI Fleet Mechanic. Analyze fleet JSON data and provide concise, prioritized maintenance advice. Warn about critical safety risks."
    )

    # Start chat session to KEEP MEMORY (History)
    chat_session = model.start_chat(history=[])
    LLM_READY = True
    print(f" [System] V3.1 AI Mechanic Ready with Memory! Active Model: {model_name}")

except Exception as e:
    print(f" [Warning] AI features disabled. API Setup error: {e}")


# --- 2. CORE CLASSES (Car & Garage) ---
class Car:
    def __init__(self, brand, model, engine, mileage):
        self.brand = brand
        self.model = model
        self.engine = engine.upper()
        self.mileage = mileage
        self.oil_limit = 10000
        self.tire_limit = 50000
        self.last_oil_change = mileage
        self.last_tire_change = mileage

    def update_mileage(self, distance):
        self.mileage += distance

    def check_oil(self):
        remains = self.oil_limit - (self.mileage - self.last_oil_change)
        return f"OK ({remains} km left)" if remains > 0 else f"WARNING (Overdue by {abs(remains)} km)"

    def check_tires(self):
        remains = self.tire_limit - (self.mileage - self.last_tire_change)
        return f"OK ({remains} km left)" if remains > 0 else f"WARNING (Overdue by {abs(remains)} km)"

    def to_dict(self):
        return {
            'brand': self.brand, 'model': self.model, 'engine': self.engine,
            'mileage': self.mileage, 'last_oil_change': self.last_oil_change,
            'last_tire_change': self.last_tire_change
        }

    @classmethod
    def from_dict(cls, data):
        car = cls(data['brand'], data['model'], data['engine'], data['mileage'])
        car.last_oil_change = data.get('last_oil_change', data['mileage'])
        car.last_tire_change = data.get('last_tire_change', data['mileage'])
        return car


class Garage:
    def __init__(self):
        self.cars = []
        self.filename = 'fleet_data.json'
        self.load_fleet()

    def add_car(self, car):
        self.cars.append(car)
        self.save_fleet()

    def save_fleet(self):
        with open(self.filename, 'w') as f:
            json.dump([car.to_dict() for car in self.cars], f, indent=4)

    def load_fleet(self):
        if os.path.exists(self.filename):
            with open(self.filename, 'r') as f:
                self.cars = [Car.from_dict(c) for c in json.load(f)]

    def get_fleet_context(self):
        return json.dumps([car.to_dict() for car in self.cars], indent=2) if self.cars else "Empty Garage."


# --- 3. AI INTEGRATION LOGIC ---
class FleetAI:
    def ask_mechanic(self, user_question, fleet_context):
        if not LLM_READY:
            return "AI is offline. Please check your API key."

        # Inject real-time data invisibly before sending the question
        prompt = f"Live Fleet Data:\n{fleet_context}\n\nUser Question: {user_question}"
        print("\nThinking...")

        # Auto-Retry Logic (Resilience against network drops)
        for attempt in range(3):
            try:
                response = chat_session.send_message(prompt)
                return response.text
            except Exception as e:
                if attempt < 2:
                    time.sleep(2) # Wait 2 seconds before retrying
                    continue
                return f"AI Communication Error: {e}"


# --- 4. MAIN MENU LOGIC ---
def main():
    garage = Garage()
    ai = FleetAI()

    while True:
        print("\n" + "="*55)
        print(f" GARAGE MANAGEMENT SYSTEM V3.1 (Fleet: {len(garage.cars)})")
        print("="*55)

        # Active Banner
        if garage.cars:
            car = garage.cars[-1] # Show the most recently added car
            print(f"   ACTIVE VEHICLE: {car.brand} {car.model}")
            print(f"  • Engine : {car.engine} | Mileage: {car.mileage:,} km")
            print(f"  • Status : Oil [{car.check_oil()}] | Tires [{car.check_tires()}]")
            print("="*55)

        print("[Menu Options]:")
        print(" 1. Add Trip to Active Car")
        print(" 2. Reset Oil (Active Car)")
        print(" 3. Reset Tires (Active Car)")
        print(" 4. Add New Car to Fleet")
        print(" 5. Fleet Health Dashboard")
        print(" 6. Ask AI Mechanic (Now with Memory) ")
        print(" 7. View AI JSON Context")
        print(" 8. Exit")

        choice = input("\nSelect (1-8): ")

        if choice == '1':
            if garage.cars:
                dist = int(input("Enter trip distance (km): "))
                garage.cars[-1].update_mileage(dist)
                garage.save_fleet()
                print(" Trip added successfully!")
            else:
                print(" Add a car first!")

        elif choice == '2':
            if garage.cars:
                garage.cars[-1].last_oil_change = garage.cars[-1].mileage
                garage.save_fleet()
                print(" Oil counter reset!")

        elif choice == '3':
            if garage.cars:
                garage.cars[-1].last_tire_change = garage.cars[-1].mileage
                garage.save_fleet()
                print(" Tires counter reset!")

        elif choice == '4':
            brand = input("Brand (e.g., Toyota): ")
            model = input("Model (e.g., RAV4): ")
            engine = input("Engine (GASOLINE/DIESEL/EV): ")
            mileage = int(input("Current Mileage: "))
            garage.add_car(Car(brand, model, engine, mileage))
            print(" Car added to fleet!")

        elif choice == '5':
            print("\n---  Fleet Health Dashboard ---")
            for i, c in enumerate(garage.cars):
                print(f"[{i+1}] {c.brand} {c.model} | Oil: {c.check_oil()} | Tires: {c.check_tires()}")

        elif choice == '6':
            q = input("\nAsk the mechanic: ")
            context = garage.get_fleet_context()
            answer = ai.ask_mechanic(q, context)
            print(f"\n Mechanic Says:\n{answer}\n")
            input("Press Enter to continue...")

        elif choice == '7':
            print("\n---  Raw JSON Data Sent to AI ---")
            print(garage.get_fleet_context())

        elif choice == '8':
            print("Exiting... Safe travels! ")
            break

        else:
            print(" Invalid choice. Please select 1-8.")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


 [System] V3.1 AI Mechanic Ready with Memory! Active Model: gemini-3-flash-preview

 GARAGE MANAGEMENT SYSTEM V3.1 (Fleet: 0)
[Menu Options]:
 1. Add Trip to Active Car
 2. Reset Oil (Active Car)
 3. Reset Tires (Active Car)
 4. Add New Car to Fleet
 5. Fleet Health Dashboard
 6. Ask AI Mechanic (Now with Memory) 
 7. View AI JSON Context
 8. Exit

Select (1-8): 4
Brand (e.g., Toyota): Toyota
Model (e.g., RAV4): RAV4
Engine (GASOLINE/DIESEL/EV): GASOLINE
Current Mileage: 15000
 Car added to fleet!

 GARAGE MANAGEMENT SYSTEM V3.1 (Fleet: 1)
   ACTIVE VEHICLE: Toyota RAV4
  • Engine : GASOLINE | Mileage: 15,000 km
  • Status : Oil [OK (10000 km left)] | Tires [OK (50000 km left)]
[Menu Options]:
 1. Add Trip to Active Car
 2. Reset Oil (Active Car)
 3. Reset Tires (Active Car)
 4. Add New Car to Fleet
 5. Fleet Health Dashboard
 6. Ask AI Mechanic (Now with Memory) 
 7. View AI JSON Context
 8. Exit

Select (1-8): 1
Enter trip distance (km): 2000
 Trip added successfully!

 GARAGE MANAGE